## Dependencies

In [6]:
## libraries
import os
import sys
import numpy as np
import pandas as pd

## project root
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

## modules
from src.evaluators.config import FEAT_X, FEAT_Z, TARGET
from src.evaluators.config import load_data, load_models
from src.evaluators.training import fit_predict_frontier
from src.evaluators.metrics import consensus_metrics
from src.evaluators.perturbate import invariant_perturb, signature_perturb

## Initalization

In [2]:
## setup
data = load_data()
models = load_models()

## Training

In [3]:
## perturbation configs
perturbations = [
    ## perturbation of graph invariants: additive noise
    {"type": "invariant", "method": "noise (0.01)", "kwargs": {"method": "noise", "noise_level": 0.01}},
    {"type": "invariant", "method": "noise (0.05)", "kwargs": {"method": "noise", "noise_level": 0.05}},
    {"type": "invariant", "method": "noise (0.10)", "kwargs": {"method": "noise", "noise_level": 0.10}},

    ## perturbation of graph invariants: feature ablation
    {"type": "invariant", "method": "subset (0.9)", "kwargs": {"method": "subset", "subset_fraction": 0.9}},
    {"type": "invariant", "method": "subset (0.7)", "kwargs": {"method": "subset", "subset_fraction": 0.7}},
    {"type": "invariant", "method": "subset (0.5)", "kwargs": {"method": "subset", "subset_fraction": 0.5}},

    ## perturbation of observations: subsampling
    {"type": "observation", "method": "subsample (0.9)", "kwargs": {"method": "subsample", "fraction": 0.9}},
    {"type": "observation", "method": "subsample (0.7)", "kwargs": {"method": "subsample", "fraction": 0.7}},
    {"type": "observation", "method": "subsample (0.5)", "kwargs": {"method": "subsample", "fraction": 0.5}},
]

## results
results = []

## iterate over models
for model_name, model in models.items():

    ## baseline frontier (train once on clean data)
    y_pred_base, _, _ = fit_predict_frontier(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET
    )

    ## iterate over perturbations
    for p in perturbations:

        ## copy data
        data_pert = data.copy()

        ## apply perturbation
        if p["type"] == "invariant":
            X_new = invariant_perturb(data_pert[FEAT_X], **p["kwargs"])
            data_pert[FEAT_X] = X_new.values

        elif p["type"] == "observation":
            data_pert, _, _ = signature_perturb(
                X = data_pert,
                Z = data_pert[FEAT_Z],
                y = data_pert[TARGET],
                **p["kwargs"]
            )
            data_pert = data_pert.reset_index(drop = True)

        ## perturbed frontier (retrain on deformed data)
        y_pred_pert, _, _ = fit_predict_frontier(
            data = data_pert,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET
        )

        ## compare baseline vs perturbed rankings
        n = min(len(y_pred_base), len(y_pred_pert))
        metrics = consensus_metrics(
            y_true = y_pred_base[:n],
            y_pred = y_pred_pert[:n]
        )

        ## store
        results.append({
            "model": model_name,
            "perturbation": p["type"],
            "method": p["method"],
            **metrics
        })

        print(f"{model_name} | {p['method']} | rho={metrics['rho']:.4f} | rbo={metrics['rbo']:.4f}")

linear_quantile | noise (0.01) | rho=1.0000 | rbo=0.9282
linear_quantile | noise (0.05) | rho=1.0000 | rbo=0.9282
linear_quantile | noise (0.10) | rho=1.0000 | rbo=0.9282
linear_quantile | subset (0.9) | rho=1.0000 | rbo=0.9282
linear_quantile | subset (0.7) | rho=1.0000 | rbo=0.9282
linear_quantile | subset (0.5) | rho=1.0000 | rbo=0.9282
linear_quantile | subsample (0.9) | rho=0.2208 | rbo=0.4532
linear_quantile | subsample (0.7) | rho=0.2304 | rbo=0.3427
linear_quantile | subsample (0.5) | rho=-0.3497 | rbo=0.2082
linear_convex | noise (0.01) | rho=1.0000 | rbo=0.9282
linear_convex | noise (0.05) | rho=1.0000 | rbo=0.9282
linear_convex | noise (0.10) | rho=1.0000 | rbo=0.9282
linear_convex | subset (0.9) | rho=1.0000 | rbo=0.9282
linear_convex | subset (0.7) | rho=1.0000 | rbo=0.9282
linear_convex | subset (0.5) | rho=1.0000 | rbo=0.9282
linear_convex | subsample (0.9) | rho=-0.1361 | rbo=0.2404
linear_convex | subsample (0.7) | rho=0.1495 | rbo=0.3397
linear_convex | subsample (0.5

## Post-Processing

In [4]:
## results
results_data = pd.DataFrame(results)
feat = ["model", "perturbation", "method", "r", "rho", "tau", "dcr", "rbo"]
results_data = results_data[feat]

## Results

In [5]:
## mean agreement by perturbation method
results_data.groupby(["perturbation", "method"])[["r", "rho", "tau", "dcr", "rbo"]].mean()

r       rho       tau       dcr       rbo
perturbation method                                                           
invariant    noise (0.01)     0.952729  0.955908  0.896423  0.958802  0.840302
             noise (0.05)     0.889383  0.947270  0.881404  0.925127  0.822950
             noise (0.10)     0.943854  0.945593  0.879879  0.947752  0.836082
             subset (0.5)     0.941548  0.947939  0.890730  0.953792  0.818104
             subset (0.7)     0.877150  0.879548  0.837425  0.905845  0.821961
             subset (0.9)     0.916527  0.895717  0.841754  0.915219  0.808026
observation  subsample (0.5)  0.174876  0.166521  0.134910  0.432692  0.327918
             subsample (0.7) -0.118652 -0.099217 -0.058499  0.374087  0.278008
             subsample (0.9)  0.151691  0.128424  0.098098  0.376105  0.330470